In [1]:
import re
import requests
from bs4 import BeautifulSoup
from dataclasses import dataclass
import sentencepiece as spm
from tqdm import tqdm
import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import lr_scheduler, AdamW
from torch.utils.data import Dataset, DataLoader

##### MINI LLaMA PROJECT
###### In this Note book I will build a Mini Llama from scratch.
###### It will be a very short 4 step small project for deeper understanding.
###### of the workflow of LlaMA, what makes it different from other models.

##### STEP 1: TOKENISATION - SPLITING RAW TEXT INTO TOKENS
###### - For this stage of the we'll train a SentencePiece tokenizer. 
###### - the link to the dataset that I will train the tokenizer on.

###### STEP 2: MODEL ARCHITEXTURE - COMPONENTS 
###### - Transformer decoder-only architecture (causal self-attention).
###### - Rotary positional embeddings (RoPE).
###### - Grouped Query attention with efficient scaling.
###### - Feed-forward layers with SwiGLU activation.
###### - RMSNorm (pre-norm style).

##### STEP 3: PRETRAINING - SETUP
###### - Loss: CrossEntropyLoss.
###### - Optimizer: AdamW with learning rate of 1e-04
###### - Learning rate schedule: cosine decay with warmup.


##### Step 4: Fine-Tuning
###### - Full fine-tuning: update all weights (expensive).
###### - Parameter-efficient: LoRA (torchtune is very useful for this process)
###### - since our model is fairly small we'll update all the weights. 

##### 1. TOKENISATION WITH SENTENCEPIECE 

###### 1.1 - looking for data. :-)

In [5]:
url = 'https://grokipedia.com/page/Andrew_Tate'
response = requests.get(url)
soup = BeautifulSoup(response.text, 'lxml')
paragraphs = [p.get_text() for p in soup.find_all('h1')]
clean_text = "\n".join(paragraphs)

print(clean_text)

Andrew Tate


In [17]:
import requests
from bs4 import BeautifulSoup

def scrape_pages(urls):
    for i, url in enumerate(urls, start=1):
        try:
            response = requests.get(url)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'lxml')
            
            print(f"--- Content from {url} ---")
            text = soup.get_text()
            print(text[:500])  
            
            filename = f"pretaining_data_{i}.txt"
            with open(filename, 'w', encoding='utf-8') as file:
                file.write(text)
                
            print(f"Saved content to {filename}\n")
        except Exception as e:
            print(f"Error scraping {url}: {e}")

urls = [
    'https://grokipedia.com/page/Christianity',
    'https://grokipedia.com/page/Jesus',
    'https://grokipedia.com/page/Therapy'
]

scrape_pages(urls)


--- Content from https://grokipedia.com/page/Christianity ---
 Christianity â Grokipedia
                      Search âK              Suggest Edit Sign in                Sign in                       Etymology and Core Identity     Etymology     Defining Characteristics as a Religion     Historical Origins and Expansion     Second Temple Judaism and Hellenistic Judaism     Persian Period (c. 538â332 BCE)     Hellenistic Period (c. 333 BCEâ70 CE)     Hasmonean Dynasty (141â37 BC)     Maccabean Revolt and the redaction of the Book of Daniel     Jew
Saved content to pretaining_data_1.txt

--- Content from https://grokipedia.com/page/Jesus ---
 Jesus â Grokipedia
                      Search âK              Suggest Edit Sign in                Sign in                       Historical and Cultural Context     First-Century Judea and Galilee     Primary Sources and Evidence     New Testament Gospels and Epistles     Extra-Biblical Written References     Archaeological and Materi

###### 1.2 - cleaning the data with regex

In [3]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    return text.strip()


In [19]:
file_names = ['pretaining_data_1.txt', 'pretaining_data_2.txt', 'pretaining_data_3.txt']
with open('pretraining_data.txt', 'w', encoding = 'utf-8') as outfile:
    for filename in file_names:
        with open(filename, 'r', encoding = 'utf-8') as file:
            outfile.write(file.read())

            outfile.write('\n')

In [24]:
with open('pretraining_data.txt', 'r', encoding = 'utf-8') as file:
    f = file.read()

cleaned_pr_data = clean_text(f)
print(f'Number of words: {len(f):,}')

Number of words: 517,415


In [26]:
with open('pretraining_data_cleaned.txt', 'w', encoding = 'utf-8') as file:
    file.write(cleaned_pr_data)

In [ ]:


with open("page_1.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

with open("page_2.txt", "r", encoding="utf-8") as f:
    raw_text_2 = f.read()

with open("page_3.txt", "r", encoding="utf-8") as f:
    raw_text_3 = f.read()

cleaned_text = clean_text(raw_text)
cleaned_text_2 = clean_text(raw_text_2)
cleaned_text_3 = clean_text(raw_text_3)

print(cleaned_text_3[:200])

ChatGPT     Grokipedia
 Search    K Suggest Edit Sign in Sign in Overview Definition and Core Functionality Initial Launch and Rapid Adoption Historical Development Origins at OpenAI Pre ChatGPT Proto


In [79]:
with open("page_1_clean.txt", "w", encoding="utf-8") as f:
   f.write(cleaned_text)
with open("page_2_clean.txt", "w", encoding="utf-8") as f:
   f.write(cleaned_text_2)
with open("page_3_clean.txt", "w", encoding="utf-8") as f:
   f.write(cleaned_text_3)


In [81]:
with open("page_1_clean.txt", "r", encoding="utf-8") as f:
    file_1 = f.read()
with open("page_2_clean.txt", "r", encoding="utf-8") as f:
    file_2 = f.read()
with open("page_3_clean.txt", "r", encoding="utf-8") as f:
    file_3 = f.read()


print(f'Number: {len(file_1):,}')
print(f'Number: {len(file_2):,}')
print(f'Number: {len(file_3):,}')

Number: 62,341
Number: 297,813
Number: 108,957


###### 1.4 - Training our tokenizer on the scrapped data.

In [84]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input=["page_1_clean.txt", "page_2_clean.txt", "page_3_clean.txt"], 
    model_prefix="llaMA_tokenizer",
    vocab_size= 10000,
    model_type="bpe",                                
    max_sentence_length=100000,                    
    character_coverage=1.0,
    byte_fallback=True,                     
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

###### 1.5 - trying out our trained encoder

In [10]:
tokenizer = spm.SentencePieceProcessor(model_file = 'llaMA_tokenizer.model')

In [6]:
text = 'Hi My name is sangunde, welcome to my github. Leave a pull request'

print(f'ENCODING\n')
pieces = tokenizer.encode(text, out_type = str)
ids = tokenizer.encode(text, out_type = int)
print(f'Pieces: {pieces}')
print(f'ids: {ids}')
print('DECODING\n')
token_ids: int = [384, 9949, 5134, 3422, 442, 267, 593, 403, 9948, 48, 290, 6083, 300, 2341, 359, 335, 551, 50, 9158, 261, 279, 1185, 862, 395]
print(tokenizer.decode(token_ids))
print(len(ids))

ENCODING

Pieces: ['▁H', 'i', '▁My', '▁name', '▁is', '▁s', 'ang', 'und', 'e', '<0x2C>', '▁w', 'elcome', '▁to', '▁my', '▁g', 'ith', 'ub', '<0x2E>', '▁Leave', '▁a', '▁p', 'ull', '▁requ', 'est']
ids: [384, 9949, 5134, 3422, 442, 267, 593, 403, 9948, 48, 290, 6083, 300, 2341, 359, 335, 551, 50, 9158, 261, 279, 1185, 862, 395]
DECODING

Hi My name is sangunde, welcome to my github. Leave a pull request
24


In [23]:
token_ids = tokenizer.encode(f, out_type = int)
print(f'Number of tokens: {len(token_ids):,}')

Number of tokens: 187,477


##### 1.1 CONFIGURATION SETUP

In [27]:
#configuration can be done with a python or dataclass 
#for this tutorial we'll use a dataclass
@dataclass
class LlaMAConfig:
    vocab_size: int = 10000
    hidden: int = 512
    context_length: int = 128
    dropout_rate: int = 0.1
    q_heads: int = 8
    n_layers: int = 4
    kv_heads: int = 4
    bias: bool = False

##### 2. MODEL ARCHITECTURE

###### 2.1 Secondly, we put in place our feed forward network.
###### With a SwiGLU activation function
###### DESCRIPTION
###### is a gated feed-forward layer. It computes two projections of the same token representation, applies the SiLU activation to one projection, applies element-wise multiplication, and projects the result back to the model.

###### more reading: https://sebastianraschka.com/faq/docs/swiglu-modern-llms.html

In [28]:
class SwiGLUFFN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.w_1 = nn.Linear(config.hidden, config.hidden, bias = config.bias)
        self.w_2 = nn.Linear(config.hidden, config.hidden, bias = config.bias)
        self.dropout = nn.Dropout(config.dropout_rate)
        self.out_proj = nn.Linear(config.hidden, config.hidden, bias = config.bias)

    def forward(self, x: torch.tensor) -> torch.tensor:
        x = F.silu(self.w_1(x)) * self.w_2(x)
        x = self.dropout(x)
        return self.out_proj(x)

###### 2.2 firstly, we fix our normalisation layer. 
###### DESCRIPTION
###### normalizes inputs based on their root mean square (RMS) rather than their mean and variance. 
###### references: https://medium.com/@VectorWorksAcademy/rmsnorm-a-computationally-efficient-normalization-technique-e5dabc3981b6

In [29]:
class RMSNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(config.hidden))
        self.eps = 1e-05

    def forward(self, x: torch.tensor) -> torch.tensor:
        mean_square = x.pow(2).mean(-1, keepdims = True)
        rms = torch.sqrt(mean_square + self.eps)
        return self.gamma + x / rms 

###### 2.3 Thirdly, we implement RoPE (Rotary Positional Encoding)
###### DESCRIPTION
###### - Instead of adding position information, RoPE rotates the query and key vectors by an angle that depends on their position.
###### - this changed the way a transformer deals with the order of the tokens. 
###### references: 

In [30]:
class RotaryPositionalEncoding(nn.Module):
    def __init__(self, head_dim, base: int = 10000):
        super().__init__()
        self.head_dim = head_dim
        self.base = base

    def rotation_angles(self, x: torch.Tensor, y: torch.Tensor, theta: torch.Tensor):
        return (x * torch.cos(theta) - y * torch.sin(theta),
                x * torch.sin(theta) + y * torch.cos(theta))

    def forward(self, pos: int, query: torch.Tensor, key: torch.Tensor):
        
        freq = self.base ** (-2 * torch.arange(0, self.head_dim // 2) / self.head_dim)
        theta = torch.outer(pos, freq)
        theta = theta.to(query.device)  
        theta = theta.unsqueeze(0).unsqueeze(0)    

        q1, q2 = query[..., 0::2], query[..., 1::2]
        k1, k2 = key[..., 0::2], key[..., 1::2]

        rq1, rq2 = self.rotation_angles(q1, q2, theta)
        rk1, rk2 = self.rotation_angles(k1, k2, theta)

        rotated_query = torch.stack([rq1, rq2], dim=-1).reshape_as(query)
        rotated_key   = torch.stack([rk1, rk2], dim=-1).reshape_as(key)

        return rotated_query, rotated_key


###### 2.4 The fourth part is the GQA (Grouped Query Attention)
###### DESCRIPTION
###### - Multiple query heads share the same K/V projections, but still have separate Q projections.
###### -Benefit: fewer parameters, faster inference, less memory usage, while keeping expressiveness. 
###### references: https://saeedmehrang.github.io/blogs/language-modeling/llm-2025-overview/grouped-query-attention/

In [31]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, num_queries, num_kv_groupes, dropout):
        super().__init__()
        assert d_model % num_queries == 0, f'{d_model} must be divible by {num_queries}'
        assert num_queries % num_kv_groupes == 0, f'{num_queries} must be divisible by {num_kv_groupes}'

        self.d_model = d_model
        self.num_queries = num_queries
        self.kv_heads = num_kv_groupes
        self.head_dim = d_model // num_queries
        self.query_per_kv = num_queries // num_kv_groupes
        self.kv_dims = self.head_dim * num_kv_groupes
        self.drop_out = dropout

        self.w_q = nn.Linear(d_model, d_model, bias = False)
        self.w_k = nn.Linear(d_model, self.kv_dims, bias = False)
        self.w_v = nn.Linear(d_model, self.kv_dims, bias = False)
        self.output_proj = nn.Linear(d_model, d_model, bias = False)

        #applying RoPE to out GQA
        self.rope = RotaryPositionalEncoding(self.head_dim)

    def forward(self, x: torch.tensor) -> torch.tensor:
        B, T, E = x.shape
        q = self.w_q(x).view(B, T, self.num_queries, self.head_dim)
        k = self.w_k(x).view(B, T, self.kv_heads, self.head_dim)
        v = self.w_v(x).view(B, T, self.kv_heads, self.head_dim)

        k = k.repeat_interleave(self.query_per_kv, dim = 2)
        v = v.repeat_interleave(self.query_per_kv, dim = 2)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        pos = torch.arange(T, device=x.device)  
        q, k = self.rope(pos=pos, query=q, key=k)
        
        context = F.scaled_dot_product_attention(q, k, v, is_causal = True, dropout_p = self.drop_out)
        context = context.transpose(1, 2).contiguous().view(B, T, E)
        return self.output_proj(context)

###### 2.5 Putting it all together in a transformer Block
###### DESCRIPTION

In [32]:
class LlamaTransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.rms_norm1 = RMSNorm(config)
        self.rms_norm2 = RMSNorm(config)
        self.attention = GroupedQueryAttention(config.hidden, 
                                               config.q_heads, 
                                               config.kv_heads, 
                                               dropout = 0.1)
        self.ffn = SwiGLUFFN(config)

    def forward(self, x: torch.tensor) -> torch.tensor:

        # RMSNorm + Attention
        residual = x
        x = self.rms_norm1(x)
        x = self.attention(x)
        x = x + residual

        # RMSNorm + Attention
        residual = x 
        x = self.rms_norm2(x)
        x = self.ffn(x)
        x = x + residual

        return x


##### 4. COMPLETE MODEL

In [33]:
class LlaMASimple(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_emb = nn.Embedding(config.vocab_size, config.hidden)
        self.dropout = nn.Dropout(config.dropout_rate)
        self.trans_block = nn.ModuleList(
            [LlamaTransformerBlock(config) for _ in range(config.n_layers)]
        )
        self.rms_norm = RMSNorm(config)
        self.output_head = nn.Linear(config.hidden, config.vocab_size, bias = False)

    def forward(self, x: torch.tensor) -> torch.tensor:
        token_emb = self.token_emb(x)
        x = self.dropout(token_emb)
        for layer in self.trans_block:
            x = layer(x)
        x = self.rms_norm(x)
        logits = self.output_head(x)
        return logits

In [34]:
llama_model = LlaMASimple(LlaMAConfig)

In [35]:
total_params = sum(params.numel() for params in llama_model.parameters())
print(f'Number of parameters: {total_params:,}M')

Number of parameters: 16,536,064M


##### 4. PRETRAINING

###### 4.1 - PREPARING THE DATASETS AND DATALOADERS 

In [36]:
class TextDataset(Dataset):
    def __init__(self, text_data, tokenizer, max_length, stride):
        super().__init__()
        self.words = []
        self.targets = []

        token_ids = tokenizer.encode(text_data)

        for i in range(0, len(token_ids) - max_length, stride):
            words = token_ids[i:i + max_length]
            targets = token_ids[i + 1: i + max_length + 1]
            self.words.append(torch.tensor(words))
            self.targets.append(torch.tensor(targets))


    def __getitem__(self, index):
        return self.words[index], self.targets[index]

    def __len__(self):
        return len(self.words)


def dataloader(txt, batch_size = 4, max_length = 256, 
               stride = 128, shuffle = True, drop_last = True):
    
    dataset = TextDataset(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size = batch_size,
        shuffle = shuffle,
        drop_last = drop_last,
    )

    return dataloader

In [39]:
with open('pretraining_data_cleaned.txt', 'r', encoding = 'utf-8') as file:
    f = file.read()

encoded  = tokenizer.encode(f)
print(f'Number of tokens: {len(encoded):,}')

Number of tokens: 115,466


In [40]:
train_loader = dataloader(f, batch_size = 2, 
                          max_length = LlaMAConfig.context_length,
                          stride = LlaMAConfig.context_length,
                          drop_last = True,
                          shuffle = True)

In [19]:
for idx, (word, target) in enumerate(train_loader):
    print(word,'-->' ,target)
    if idx == 0:
        break

tensor([[  50, 1239,  301, 9602,   48,  330, 3026,  859,   48,  497, 6500, 6649,
          750,  383,  537,  297,   63,  764,  497,  704, 9527,  367,  268, 9950,
          677,  518,  995,  262, 9986,  265,  287, 6152, 9948,  297, 9960,   49,
         4224, 2552,  279, 3339,  592,  341,  261, 4590,  436,  295, 8439, 2012,
          301, 2460,  395, 2159,   50, 9947,   38, 8407,  784,   43, 9953, 1480,
         2443, 1276,  442,  300, 6985,  516, 9561,  947, 4748, 1437,   48,   38,
          704, 1014,  315,  301,  261, 9978, 9949,  308, 9953,  497,  615,  342,
         1224, 1201,  301,  311,  550, 3152,  287,  267,  298,  372,  315,  411,
          374, 1370, 5946, 6544, 7380, 9948,  262,   49, 9951,  510,   48, 1604,
           48,  331,  261, 1228, 2828,   48,  330, 1450,  901, 1167,  492,  420,
         1945, 4605, 8636,  561,   63,  287,  294, 9955],
        [5230,   50,  488,  585,  446,  515, 1429,  315,  301,  367,  319, 1298,
         9950,  296, 2254, 6281,  300,  552, 6051, 

###### 4.2 - hyperparameters

In [42]:
lr = 0.0004
epochs = 50
optimizer = AdamW(llama_model.parameters(), lr)
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max = 10)

###### 4.3 - Text generation

In [43]:
def text_generation(model,
                    idx,
                    max_new_token, 
                    context_size,
                    temperature = 0.0,
                    eos_id = None,
                    k = 30):
    
    for _ in range(max_new_token):
        idx_cond = idx[:, -context_size:]
        with torch.inference_mode():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if k is not None:
            top_logits, _ = torch.topk(logits, k)
            logits = torch.where(
                condition = logits < top_logits[:, -1],
                input = torch.tensor(-torch.inf),
                other = logits
            )

        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, -1)
            id_next = torch.multinomial(probs, num_samples = 1)
        else:
            id_next = torch.argmax(logits, -1, keepdim = True)
        if id_next == eos_id:
            break
        idx = torch.concat([idx, id_next], dim = -1)
    return idx

###### 4.4 Utility functions

In [44]:
def calc_loss_batch(input_batch, output_batch, model):
    logits = model(input_batch)
    loss = F.cross_entropy(logits.flatten(0, 1), output_batch.flatten())
    return loss 


def text_to_ids(text, tokenizer):
    encoded = tokenizer.encode(text) 
    encoded = torch.tensor(encoded).unsqueeze(0)
    return encoded

def ids_to_text(ids, tokenizer):
    ids = ids.squeeze(0).detach().tolist()
    return tokenizer.decode(ids)


def generate_text(model, tokenizer, start_context):
    model.eval()
    context_size = LlaMAConfig.context_length
    encoded = text_to_ids(start_context, tokenizer)
    with torch.inference_mode():
        token_ids = text_generation(model, encoded, max_new_token=20, context_size = context_size, temperature = 0.5, k = 30)

    decode = ids_to_text(token_ids, tokenizer)
    decoded_text = decode.replace('\n', ' ')
    print(decoded_text)
    model.train()

In [48]:
text = 'Every morning is great'
encoded = tokenizer.encode(text)
encoded = torch.tensor(encoded).unsqueeze(0)
encoded

tensor([[3444, 6974,  689,  442, 2606]])

In [50]:
new_text = text_generation(llama_model, encoded, 10, LlaMAConfig.context_length, temperature = 0.8)
tokenizer.decode(new_text.squeeze(0).detach().tolist())

'Every morning is great misleading capabilitieston argued highly BTC rh philosophyosp members'

###### 4.5 - Training loop

In [51]:
def training_loop(
        model,
        train_loader,
        num_epochs,
        lr_scheduler,
        optimizer
):
    pbar = tqdm(range(num_epochs), desc = 'Training') 

    for _ in pbar:
        model.train()
        training_loss = 0.0

        for word_batch, target_batch in train_loader:
            optimizer.zero_grad(set_to_none = True)
            loss = calc_loss_batch(word_batch, target_batch, model)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0)
            optimizer.step()
            training_loss += loss.item()

        lr_scheduler.step()

        pbar.set_postfix(loss=f'{training_loss / len(train_loader)}')
        generate_text(model, tokenizer, 'Maybe I should')


In [52]:
training_loop(llama_model, train_loader, epochs, scheduler, optimizer)

Training:   2%|▏         | 1/50 [01:45<1:26:09, 105.50s/it, loss=6.539026565932382]

Maybe I should in the God s Baptistles of the Holy Spiritity for the God


Training:   4%|▍         | 2/50 [03:38<1:27:57, 109.94s/it, loss=5.027601131050127]

Maybe I should a Jewish Christian communities and the Messiah to the Gospel of the Son


Training:   6%|▌         | 3/50 [05:28<1:26:09, 109.99s/it, loss=4.258942156831864]

...

Maybe I should be interpreted within the Church of the East under the East following the Council of Eph


Training:  98%|█████████▊| 49/50 [1:51:11<02:32, 152.31s/it, loss=0.0345642662435737]  

Maybe I should be Resurrection The Case for And Against His RisingA Scientist Looks


Training: 100%|██████████| 50/50 [1:53:44<00:00, 136.49s/it, loss=0.02810908964899123]

Maybe I should be interpreted within the Church of the East East Syriac Nestorian


In [ ]:
gen_ids = text_generation(llama_model, 
                idx = text_to_ids('Every effort', tokenizer),
                max_new_token = 50, 
                context_size = LlaMAConfig.context_length,
                temperature = 0.8,
                k = 20)

In [62]:
gen_text = ids_to_text(gen_ids, tokenizer)
print(gen_text)

Every effort in which Christian and institutions in Mor sent primacy and a chartered Europe branches Catholic I Were Archaeological and Material Corroborations Timeline of Surviving Sources Assessment of Histor


In [64]:
torch.save(llama_model.state_dict(), 'llama_model.pth')

##### 4. INSTRUCTION FINE TUNING